# 03 - Modélisation, Évaluation et Explicabilité

Ce notebook couvre les phases **Modélisation** et **Évaluation** de CRISP-DM. Le script exécutable principal est `src/train.py`; le notebook explique comment inspecter les sorties.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import json
import pandas as pd
from IPython.display import display, Image

## Entraînement des Modèles

Le script entraîne Régression Logistique, Random Forest et XGBoost avec GridSearchCV et validation croisée à 5 plis.

```powershell
python src/train.py
```

In [ ]:
# Dans un environnement notebook local, décommenter pour entraîner depuis le notebook.
# import subprocess
# subprocess.run([sys.executable, str(ROOT / 'src' / 'train.py')], check=True)

## Tableau de Comparaison

Le fichier `outputs/model_metrics.csv` contient AUC-ROC, F1-score, précision, rappel et meilleurs hyperparamètres.

In [ ]:
metrics_path = ROOT / 'outputs' / 'model_metrics.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    print('Exécuter src/train.py pour générer model_metrics.csv')

## Rapports de Classification

Les rapports détaillent précision, rappel, F1-score et support. Le rappel est particulièrement important car les faux négatifs sont des churners manqués.

In [ ]:
reports_path = ROOT / 'outputs' / 'classification_reports.json'
if reports_path.exists():
    reports = json.loads(reports_path.read_text())
    print(json.dumps(reports[0]['classification_report'], indent=2)[:3000])
else:
    print('Exécuter src/train.py pour générer les rapports')

## Courbes et Matrices de Confusion

Les courbes ROC, précision-rappel et matrices de confusion sont générées dans `reports/figures`.

In [ ]:
fig_dir = ROOT / 'reports' / 'figures'
if fig_dir.exists():
    for path in sorted(fig_dir.glob('*roc_curve.png')):
        display(Image(filename=str(path)))
else:
    print('Exécuter src/train.py pour générer les figures')

## Optimisation du Seuil

Le seuil 0.50 n’est pas forcément optimal. Le fichier `threshold_optimization.csv` compare plusieurs seuils et permet d’équilibrer rappel, précision et faux négatifs.

In [ ]:
threshold_path = ROOT / 'outputs' / 'threshold_optimization.csv'
if threshold_path.exists():
    display(pd.read_csv(threshold_path).head(10))
else:
    print('Exécuter src/train.py pour générer threshold_optimization.csv')

## Importance des Variables et SHAP

L’importance des variables identifie les drivers de churn. SHAP permet d’expliquer l’impact des variables pour XGBoost.

In [ ]:
importance_path = ROOT / 'outputs' / 'feature_importance.csv'
if importance_path.exists():
    display(pd.read_csv(importance_path).head(20))

shap_path = ROOT / 'outputs' / 'shap_feature_impact.csv'
if shap_path.exists():
    display(pd.read_csv(shap_path).head(20))

## Sortie de Déploiement

Le fichier `outputs/predictions_churn.csv` contient `customer_id`, `churn_probability`, `predicted_class` et `risk_level`. Il peut être utilisé par le CRM.

In [ ]:
pred_path = ROOT / 'outputs' / 'predictions_churn.csv'
if pred_path.exists():
    display(pd.read_csv(pred_path).head(20))
else:
    print('Exécuter src/train.py pour générer les prédictions')

## Conclusion

Le meilleur modèle est sélectionné selon l’AUC-ROC et la pertinence métier. L’analyse des faux négatifs et le choix du seuil doivent être alignés avec la capacité de campagne.